# Training-set support router — 300-paper audit

This notebook is a **read-only sampler**. It runs the same deterministic code
that the full builder imports, over the same `PAPERS=300`, `SEED=1` sample.

The routing question is now the right one:

1. Is the target supported by all source spans the extractor supplied?
2. If that quote bundle is too narrow, can a compact supporting bundle be found
   elsewhere in the **same paper**?
3. If yes, widen the open-book context. If no, quarantine the target as
   `QUARANTINE_UNVERIFIED`; do not manufacture a refusal or silently send it
   closed-book.

Closed-book examples are selected only from paper-verified targets. FACTUAL and
numerical African research knowledge are intentionally eligible: MUFASA is a
foundation model that should know the research, not only a RAG assistant.

No API, LLM, embedding download, or vector database is used here.

In [21]:
# ======================== controls and sample ==========================
import importlib
import os
import random
import sys
import textwrap
import warnings
from collections import Counter
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path

import pandas as pd

warnings.filterwarnings("ignore")
DATA = next(
    (folder for start in [Path.cwd(), *Path.cwd().parents]
     for folder in (start, start / "01-data-engineering" / "data-extraction")
     if (folder / "mufasa_corpus" / "parsed" / "markdown").is_dir()),
    None,
)
if DATA is None:
    raise FileNotFoundError("Could not locate data-extraction/mufasa_corpus")
sys.path.insert(0, str(DATA))
import mufasa_dataset as funnel
importlib.reload(funnel)

EXTRACTION = DATA / "extraction_output"
MARKDOWN = DATA / "mufasa_corpus" / "parsed" / "markdown"
PAPERS = 300
SHOW = 5
SEED = 1
ROUTER_WORKERS = min(12, max(1, (os.cpu_count() or 2) - 1))
MAX_EVIDENCE_SPANS = 3
MAX_EVIDENCE_CHARS = 8_000

tables = funnel.load_tables(EXTRACTION)
all_papers = sorted(tables["training_pairs"].paper_id.unique())
SUBSET = set(random.Random(SEED).sample(all_papers, min(PAPERS, len(all_papers))))
pairs = tables["training_pairs"]
pairs = pairs[pairs.paper_id.isin(SUBSET)].copy()
TABLE_EVIDENCE = tables["training_evidence"]
status = tables["extraction_status"]
profiles = tables["paper_profiles"]
COMPLETE = set(status[status.complete.astype("boolean").fillna(False)].paper_id)
NOT_REAL, NOT_AFRICA = funnel.failed_verdicts(profiles)
PROFILES = profiles.drop_duplicates("paper_id").set_index("paper_id").to_dict("index")
CONTEXTS = (tables["study_contexts"].sort_values("source_task")
            .drop_duplicates("paper_id").set_index("paper_id").to_dict("index"))
INNOVATION = (tables["african_innovation"].drop_duplicates("paper_id")
              .set_index("paper_id").to_dict("index"))

def wrap(label, value, width=94):
    body = textwrap.fill(str(value), width=width,
                         initial_indent="      ", subsequent_indent="      ")
    print(f"   {label}\n{body}")

def peek(rows, renderer, seed=SEED, show=SHOW):
    rows = list(rows)
    if not rows:
        print("   (nothing matched)")
        return
    for number, item in enumerate(random.Random(seed).sample(rows, min(show, len(rows))), 1):
        print(f"\n{'-' * 96}\n[{number}]")
        renderer(item)
    print(f"\n(SEED={seed}; change it only when you want a different audit sample)")

print(f"working sample: {len(SUBSET)} papers, {len(pairs):,} pairs")
print("pair types:", dict(pairs.pair_type.value_counts()))
print(f"router: {ROUTER_WORKERS} processes; same-paper lexical search only")

working sample: 300 papers, 14,887 pairs
pair types: {'REASONING': np.int64(6004), 'FACTUAL': np.int64(5934), 'RERANKER': np.int64(1477), 'PREFERENCE': np.int64(1472)}
router: 7 processes; same-paper lexical search only


In [22]:
# =============== stage 0: recover ALL evidence shapes ==================
RECOVERED = funnel.recover_evidence_bundles(EXTRACTION / "raw", papers=SUBSET)
INITIAL = {
    pair_id: funnel.combined_evidence(pair_id, TABLE_EVIDENCE, RECOVERED)
    for pair_id in pairs.pair_id
}

table_orphans = pairs[~pairs.pair_id.isin(TABLE_EVIDENCE)]
gained = [row for row in table_orphans.itertuples() if INITIAL[row.pair_id]]
multi = sum(len(bundle) > 1 for bundle in INITIAL.values())
extra = sum(max(0, len(bundle) - 1) for bundle in INITIAL.values())
shape_counts = Counter(
    shape for row in gained for shape in RECOVERED[row.pair_id]["shapes"]
)

print(f"pairs without table evidence : {len(table_orphans):,}")
print(f"recovered from raw JSON      : {len(gained):,} "
      f"({100 * len(gained) / max(len(table_orphans), 1):.1f}%)")
print(f"pairs with multiple spans    : {multi:,}  ({extra:,} spans formerly hidden)")
print("recovered shapes:", dict(shape_counts))

def show_recovery(row):
    print(f"   {row.pair_id}  [{row.pair_type}]")
    wrap("question:", row.question)
    print("   shapes:", RECOVERED[row.pair_id]["shapes"])
    for number, span in enumerate(INITIAL[row.pair_id], 1):
        wrap(f"span {number}:", span.get("quote", "")[:420])

peek(gained, show_recovery)

pairs without table evidence : 4,079
recovered from raw JSON      : 2,894 (70.9%)
pairs with multiple spans    : 420  (473 spans formerly hidden)
recovered shapes: {'flattened onto the pair': 2779, 'positive_evidence': 80, 'chosen_evidence': 20, 'nested dict': 20}

------------------------------------------------------------------------------------------------
[1]
   W2047097759:reasoning:R5  [REASONING]
   question:
      What is the scientific premise for expecting a relationship between ridge patterns and
      diabetes even though TFRC results have been contradictory?
   shapes: ['flattened onto the pair']
   span 1:
      Although various methods of Total Finger Ridge Count (TFRC) have been reported with
      contradicting results, researchers have been able to demonstrate that the ridge patterns
      are affected one way or the other by diabetes mellitus (Igbigbi *et al*., 2001).

------------------------------------------------------------------------------------------------
[

In [23]:
# ============ stage 1: structural gates and duplicate safety ===========
REASONS = {
    row.Index: funnel.discard_reasons(row, COMPLETE, NOT_REAL, NOT_AFRICA)
    for row in pairs.itertuples()
}
doomed = [row for row in pairs.itertuples() if REASONS[row.Index]]
alive = pairs.loc[[not REASONS[index] for index in pairs.index]].copy()
alive, duplicate_drops = funnel.resolve_duplicates(alive)

print(f"rows examined             : {len(pairs):,}")
print(f"hard structural discards  : {len(doomed):,}")
print("reasons:", dict(Counter(reason for values in REASONS.values() for reason in values)))
print(f"duplicate rows removed    : {len(duplicate_drops):,}")
print("  exact copies collapse; every unresolved conflicting-ID row is removed")
print(f"rows entering support gate: {len(alive):,}")

def show_doomed(row):
    print(f"   {row.pair_id}  [{row.pair_type}]")
    wrap("question:", row.question or "(blank)")
    wrap("target:", funnel.assistant_turn(row)[:300] or "(blank)")
    print("   DISCARD:", "; ".join(REASONS[row.Index]))

peek(doomed, show_doomed)

rows examined             : 14,887
hard structural discards  : 547
reasons: {'task was truncated': 174, 'blank question': 171, 'blank positive quote': 70, 'blank answer': 121, 'placeholder ellipsis in target': 12, 'blank chosen': 40, 'blank rejected': 40, 'wrong language': 10, 'judged not africa relevant': 50}
duplicate rows removed    : 8
  exact copies collapse; every unresolved conflicting-ID row is removed
rows entering support gate: 14,332

------------------------------------------------------------------------------------------------
[1]
   W2511537934:reranker:SF-022933-RR02  [RERANKER]
   question:
      Which retrieved statement best answers: Why was λρ chosen ahead of ρ and Ip as the most
      sensitive fluid discriminator?
   target:
      Most relevant: this statement gives the Gassmann modeling result that λρ separates
      0%–100% oil and brine saturations, whereas ρ and Ip are only reliable above 75% oil
      saturation.
   DISCARD: blank positive quote

------------

In [24]:
# ===== stage 2: full support router over SFT + PREFERENCE targets ===========
# One process owns one paper at a time, reads it once, and routes all its
# targets. This gives real CPU parallelism while preserving paper isolation.
sft_rows = [row for row in alive.itertuples()
            if row.pair_type in ("FACTUAL", "REASONING")]
pref_rows = [row for row in alive.itertuples() if row.pair_type == "PREFERENCE"]
support_frame = alive[alive.pair_type.isin(["FACTUAL", "REASONING", "PREFERENCE"])]
by_paper = {
    paper_id: group.to_dict("records")
    for paper_id, group in support_frame.groupby("paper_id", sort=False)
}

ALL_ROUTES = {}
with ProcessPoolExecutor(max_workers=ROUTER_WORKERS) as workers:
    futures = {}
    for paper_id, records in by_paper.items():
        ids = [record["pair_id"] for record in records]
        initial = {pair_id: INITIAL.get(pair_id, []) for pair_id in ids}
        future = workers.submit(
            funnel.route_paper_records,
            records,
            initial,
            str(MARKDOWN),
            MAX_EVIDENCE_SPANS,
            MAX_EVIDENCE_CHARS,
        )
        futures[future] = paper_id
    try:
        from tqdm.auto import tqdm
        completed = tqdm(as_completed(futures), total=len(futures), unit="paper",
                         desc="support routing")
    except ImportError:
        completed = as_completed(futures)
    for future in completed:
        ALL_ROUTES.update(future.result())

SFT_IDS = {row.pair_id for row in sft_rows}
PREF_IDS = {row.pair_id for row in pref_rows}
ROUTES = {pair_id: result for pair_id, result in ALL_ROUTES.items()
          if pair_id in SFT_IDS}
PREF_ROUTES = {pair_id: result for pair_id, result in ALL_ROUTES.items()
               if pair_id in PREF_IDS}

route_counts = Counter(result["route"] for result in ROUTES.values())
verified = route_counts["OPEN_AS_IS"] + route_counts["OPEN_WIDENED"]
print(f"SFT candidates routed : {len(sft_rows):,}")
for name in ("OPEN_AS_IS", "OPEN_WIDENED", "QUARANTINE_UNVERIFIED"):
    count = route_counts[name]
    print(f"  {name:<20} {count:>6,}  ({100 * count / max(len(sft_rows), 1):5.1f}%)")
print(f"source-supported total: {verified:,} ({100 * verified / max(len(sft_rows), 1):.1f}%)")
print("quarantine reasons:", dict(Counter(
    result["report"]["reason"] for result in ROUTES.values()
    if result["route"] == "QUARANTINE_UNVERIFIED"
)))

ROW = {row.pair_id: row for row in sft_rows}
def show_route(pair_id):
    row, result = ROW[pair_id], ROUTES[pair_id]
    print(f"   {pair_id}  [{row.pair_type}]  -> {result['route']}")
    wrap("question:", row.question)
    wrap("target:", funnel.assistant_turn(row)[:420])
    print(f"   bundle: {len(result['bundle'])} span(s), "
          f"{sum(len(s.get('quote', '')) for s in result['bundle']):,} chars")
    for number, span in enumerate(result["bundle"], 1):
        wrap(f"source {number}:", span.get("quote", "")[:360])
    if not result["paper_verified"]:
        print("   audit reason:", result["report"]["reason"])
        print("   missing figures:", result["report"]["missing_numbers"][:8])

for route_name in ("OPEN_AS_IS", "OPEN_WIDENED", "QUARANTINE_UNVERIFIED"):
    print(f"\n{'=' * 96}\n{route_name} examples")
    peek([pair_id for pair_id, result in ROUTES.items()
          if result["route"] == route_name], show_route)


support routing:   0%|          | 0/293 [00:00<?, ?paper/s]

SFT candidates routed : 11,538
  OPEN_AS_IS            2,224  ( 19.3%)
  OPEN_WIDENED          1,723  ( 14.9%)
  QUARANTINE_UNVERIFIED  7,591  ( 65.8%)
source-supported total: 3,947 (34.2%)
quarantine reasons: {'figure is not bound to the asked metric': 398, 'evidence negates the positive target': 205, 'one or more target sentences lack source support': 1700, 'figure is not bound to the asked entity': 2004, 'named term absent from evidence': 405, 'asked entity is not bound to the evidence': 417, 'negation absent from evidence': 617, 'unit mismatch': 319, 'causal relation absent from evidence': 783, 'figure absent from evidence': 674, 'causal claim supported only by association': 40, 'decrease/increase conflict': 7, 'evidence is not relevant to the question': 18, 'increase/decrease conflict': 4}

OPEN_AS_IS examples

------------------------------------------------------------------------------------------------
[1]
   W2766255033:reasoning:R-17  [REASONING]  -> OPEN_AS_IS
   question:


In [25]:
# ============ stage 3: mixed open/closed foundation curriculum =========
DESCRIPTORS = {
    paper_id: funnel.descriptor(paper_id, CONTEXTS, PROFILES, INNOVATION)
    for paper_id in SUBSET
}
VERIFIED_ROWS = [ROW[pair_id] for pair_id, result in ROUTES.items()
                 if result["paper_verified"]]
CLOSED_READY = {
    row.pair_id: funnel.verified_closed_ready(
        row, DESCRIPTORS.get(row.paper_id, ""), ROUTES[row.pair_id]["paper_verified"]
    )
    for row in VERIFIED_ROWS
}

# One mixed SFT, not two separate models/datasets. A small deliberate DUAL lane
# teaches that the same knowledge can be recalled or grounded. If a row cannot
# form a clean closed-book example, it remains open-book regardless of its hash.
ASSIGNMENT = {}
for row in VERIFIED_ROWS:
    mode = funnel.curriculum_mode(row.pair_id, 0.45, 0.45, 0.10, seed=SEED)
    if not CLOSED_READY[row.pair_id][0] and mode in ("CLOSED", "DUAL"):
        mode = "OPEN"
    ASSIGNMENT[row.pair_id] = mode

print(f"verified targets          : {len(VERIFIED_ROWS):,}")
print(f"closed-book eligible      : {sum(ok for ok, _ in CLOSED_READY.values()):,}")
print("planned mixed-SFT rows    :", dict(Counter(ASSIGNMENT.values())))
print("final production balancing should use TOKENS, not row counts")
print("closed refusals:", dict(Counter(
    why for ok, why in CLOSED_READY.values() if not ok
)))

def show_open(pair_id):
    row = ROW[pair_id]
    example = funnel.render_open(row, ROUTES[pair_id]["bundle"])
    print(f"   {pair_id} -> OPEN ({ROUTES[pair_id]['route']})")
    for turn in example["messages"]:
        wrap(f"{turn['role']}:", turn["content"][:650])

def show_closed(pair_id):
    row = ROW[pair_id]
    example = funnel.render_closed(row, DESCRIPTORS[row.paper_id])
    print(f"   {pair_id} -> CLOSED knowledge")
    for turn in example["messages"]:
        wrap(f"{turn['role']}:", turn["content"][:650])

print("\nOPEN examples")
peek([key for key, mode in ASSIGNMENT.items() if mode in ("OPEN", "DUAL")], show_open)
print("\nCLOSED examples")
peek([key for key, mode in ASSIGNMENT.items() if mode in ("CLOSED", "DUAL")], show_closed)

verified targets          : 3,947
closed-book eligible      : 3,663
planned mixed-SFT rows    : {'OPEN': 1907, 'DUAL': 358, 'CLOSED': 1682}
final production balancing should use TOKENS, not row counts
closed refusals: {'descriptor reveals the target': 197, 'single case is not a stable closed-book target': 38, 'no titled study descriptor': 49}

OPEN examples

------------------------------------------------------------------------------------------------
[1]
   W2766255033:reasoning:R-02 -> OPEN (OPEN_WIDENED)
   user:
      You are a research assistant for African scientific literature. Answer using only the
      evidence provided. If the evidence does not contain the answer, say so plainly.
      Evidence 1 (page 2; Introduction): Schistosomiasis is a public health problem in low
      resource poor countries of the world due to lack of access to health facilities and safe
      water, poor hygiene and sanitation. The disease is caused by *Schistosoma* spp.
      *Schistosma mansoni*

In [26]:
# ================== stage 4: DPO and refusal quarantine ================
# Preference chosen answers were routed in the same paper-level pass above.
# The rejected answer is preserved as contrast and never added to SFT.
DPO_CHECK = {
    row.pair_id: funnel.preference_ready(row, PREF_ROUTES[row.pair_id]["bundle"])
    for row in pref_rows
}
dpo_ready = [row for row in pref_rows
             if PREF_ROUTES[row.pair_id]["paper_verified"]
             and DPO_CHECK[row.pair_id][0]]
print(f"preference candidates : {len(pref_rows):,}")
print(f"source-supported DPO : {len(dpo_ready):,}")
print("routes:", dict(Counter(result["route"] for result in PREF_ROUTES.values())))
print("DPO contract refusals:", dict(Counter(why for ok, why in DPO_CHECK.values() if not ok)))

# Existing RERANKER hard negatives are not rendered as refusals. The old
# notebook showed at least one passage about longitudinal data that directly
# answered the causal-limitations question. These remain a separate candidate
# pool until a true non-answer validator is implemented and audited.
reranker = alive[alive.pair_type.eq("RERANKER")]
print(f"\nreranker rows quarantined from refusal SFT: {len(reranker):,}")
print("reason: hard_negative_quote is not yet proven to be a non-answer")

def show_dpo(row):
    result = PREF_ROUTES[row.pair_id]
    print(f"   {row.pair_id} -> {result['route']}")
    wrap("question:", row.question)
    wrap("chosen:", row.chosen[:300])
    wrap("rejected:", row.rejected[:300])
    for number, span in enumerate(result["bundle"], 1):
        wrap(f"source {number}:", span.get("quote", "")[:320])

peek(dpo_ready, show_dpo)


preference candidates : 1,409
source-supported DPO : 565
routes: {'OPEN_AS_IS': 212, 'QUARANTINE_UNVERIFIED': 838, 'OPEN_WIDENED': 359}
DPO contract refusals: {'chosen is not supported by the evidence': 836, 'chosen and rejected are the same': 8}

reranker rows quarantined from refusal SFT: 1,385
reason: hard_negative_quote is not yet proven to be a non-answer

------------------------------------------------------------------------------------------------
[1]
   W2323875382:preference:preference_04 -> OPEN_AS_IS
   question:
      Why was Sango-Ota chosen for this study?
   chosen:
      The research aimed at screening the possible presence of pharmaceuticals in water
      sourced from Sango-Ota, a high pharmaceutical industrial community in Ogun State,
      Nigeria.
   rejected:
      Sango-Ota was chosen because pharmaceutical pollution there had already been confirmed.
   source 1:
      This research is aimed at screening the possible presence of pharmaceuticals in water
      s

In [27]:
# ============================= final audit ==============================
hard_discards = len(doomed) + len(duplicate_drops)
support_discards = route_counts["QUARANTINE_UNVERIFIED"]
open_renderings = sum(mode in ("OPEN", "DUAL") for mode in ASSIGNMENT.values())
closed_renderings = sum(mode in ("CLOSED", "DUAL") for mode in ASSIGNMENT.values())

summary = pd.DataFrame([
    ("input papers", len(SUBSET)),
    ("input pairs", len(pairs)),
    ("hard structural/duplicate discards", hard_discards),
    ("factual + reasoning routed", len(sft_rows)),
    ("open as supplied", route_counts["OPEN_AS_IS"]),
    ("rescued by honest wider context", route_counts["OPEN_WIDENED"]),
    ("unverified and quarantined", support_discards),
    ("deterministically source-supported SFT targets", len(VERIFIED_ROWS)),
    ("planned open renderings", open_renderings),
    ("planned closed renderings", closed_renderings),
    ("deterministically source-supported DPO candidates", len(dpo_ready)),
    ("reranker/refusal candidates quarantined", len(reranker)),
], columns=["measure", "count"])
display(summary)

print("\nInterpretation")
print("- OPEN_WIDENED is salvaged data, not a relaxed pass: the prompt receives the supporting source bundle.")
print("- QUARANTINE_UNVERIFIED means this deterministic pass could not prove support; it is not labelled fabrication.")
print("- This sampler publishes nothing; the production builder calls this router over the frozen upstream-cleared corpus.")
print("- CLOSED examples come only from deterministically source-supported targets and intentionally retain African research facts/numbers.")
print("- Open and closed are one mixed SFT curriculum; DPO remains a separate objective/file.")
print("- No automatic refusal examples are emitted from the current unvalidated hard-negative pool.")

,measure,count
0,input papers,300
1,input pairs,14887
2,hard structural/duplicate discards,555
3,factual + reasoning routed,11538
4,open as supplied,2224
5,rescued by honest wider context,1723
6,unverified and quarantined,7591
7,deterministically source-supported SFT targets,3947
8,planned open renderings,2265
9,planned closed renderings,2040



Interpretation
- OPEN_WIDENED is salvaged data, not a relaxed pass: the prompt receives the supporting source bundle.
- QUARANTINE_UNVERIFIED means this deterministic pass could not prove support; it is not labelled fabrication.
- This sampler publishes nothing; the production builder calls this router over the frozen upstream-cleared corpus.
- CLOSED examples come only from deterministically source-supported targets and intentionally retain African research facts/numbers.
- Open and closed are one mixed SFT curriculum; DPO remains a separate objective/file.
- No automatic refusal examples are emitted from the current unvalidated hard-negative pool.
